# Construir la Red Neuronal

Las redes neuronales están compuestas por capas/módulos que realizan operaciones sobre los datos. El namespace `torch.nn` proporciona todos los bloques de construcción que necesitas para construir tu propia red neuronal. 

Cada módulo en PyTorch hereda de `nn.Module`. Una red neuronal es un módulo en sí misma que consiste en otros módulos (capas). Esta estructura anidada permite construir y gestionar arquitecturas complejas fácilmente.

En las siguientes secciones, construiremos una red neuronal para clasificar imágenes en el dataset FashionMNIST.

## Conceptos Clave

- **`nn.Module`**: Clase base para todos los módulos de red neuronal
- **Capas (Layers)**: Componentes individuales que transforman datos
- **Forward pass**: El proceso de pasar datos a través de la red
- **Parámetros**: Pesos y sesgos que se aprenden durante el entrenamiento

In [ ]:
import os
import torch
from torch import nn
from torch.utils.data import DataLoader
from torchvision import datasets, transforms

## Obtener Dispositivo para Entrenamiento

Queremos poder entrenar nuestro modelo en un acelerador como **CUDA** (GPU NVIDIA), **MPS** (GPU Apple), **MTIA** o **XPU**. Si el acelerador actual está disponible, lo usaremos. De lo contrario, usamos la CPU.

### ¿Por qué es importante?

- **GPU/Aceleradores**: Hasta 100x más rápido que CPU para entrenamiento de redes neuronales
- **CPU**: Más lento pero siempre disponible como respaldo
- **Memoria**: Los aceleradores tienen su propia memoria, debemos mover los datos allí

In [ ]:
device = torch.accelerator.current_accelerator().type if torch.accelerator.is_available() else "cpu"
print(f"Using {device} device")

## Definir la Clase

Definimos nuestra red neuronal heredando de `nn.Module`, e inicializamos las capas de la red neuronal en `__init__`. Cada subclase de `nn.Module` implementa las operaciones sobre los datos de entrada en el método `forward`.

### Arquitectura de nuestra red:

```
Entrada: Imagen 28×28 (784 píxeles)
    ↓
Flatten: Convierte matriz 2D a vector 1D
    ↓
Capa Linear 1: 784 → 512 neuronas
    ↓
ReLU: Función de activación
    ↓
Capa Linear 2: 512 → 512 neuronas
    ↓
ReLU: Función de activación
    ↓
Capa Linear 3: 512 → 10 neuronas (una por clase)
    ↓
Salida: 10 logits (valores sin procesar)
```

In [ ]:
class NeuralNetwork(nn.Module):
    def __init__(self):
        super().__init__()
        self.flatten = nn.Flatten()
        self.linear_relu_stack = nn.Sequential(
            nn.Linear(28*28, 512),
            nn.ReLU(),
            nn.Linear(512, 512),
            nn.ReLU(),
            nn.Linear(512, 10),
        )

    def forward(self, x):
        x = self.flatten(x)
        logits = self.linear_relu_stack(x)
        return logits

Creamos una instancia de `NeuralNetwork`, la movemos al dispositivo, e imprimimos su estructura.

In [ ]:
model = NeuralNetwork().to(device)
print(model)

## Usar el Modelo

Para usar el modelo, le pasamos los datos de entrada. Esto ejecuta el `forward` del modelo, junto con algunas operaciones en segundo plano. **¡No llames a `model.forward()` directamente!**

Llamar al modelo con la entrada devuelve un tensor bidimensional con:
- **dim=0**: correspondiente a cada salida de 10 valores predichos sin procesar para cada clase
- **dim=1**: correspondiente a los valores individuales de cada salida

Obtenemos las probabilidades de predicción pasándolo a través de una instancia del módulo `nn.Softmax`.

In [ ]:
X = torch.rand(1, 28, 28, device=device)
logits = model(X)
pred_probab = nn.Softmax(dim=1)(logits)
y_pred = pred_probab.argmax(1)
print(f"Predicted class: {y_pred}")

## Capas del Modelo

Desglosemos las capas en el modelo FashionMNIST. Para ilustrarlo, tomaremos un minibatch de muestra de 3 imágenes de tamaño 28x28 y veremos qué le sucede al pasar a través de la red.

In [ ]:
input_image = torch.rand(3,28,28)
print(input_image.size())

### nn.Flatten

Inicializamos la capa `nn.Flatten` para convertir cada imagen 2D de 28x28 en un array contiguo de 784 valores de píxeles (la dimensión del minibatch en dim=0 se mantiene).

**Función:** Transforma tensores multidimensionales en vectores 1D, preservando la dimensión del batch.

**Parámetros:**
- `start_dim=1`: dimensión desde donde comenzar el aplanamiento
- `end_dim=-1`: dimensión hasta donde aplanar (por defecto, la última)

In [ ]:
flatten = nn.Flatten()
flat_image = flatten(input_image)
print(flat_image.size())

### nn.Linear

La capa lineal es un módulo que aplica una transformación lineal sobre la entrada usando sus pesos y sesgos almacenados.

**Función:** Realiza la operación $y = xW^T + b$

**Parámetros:**
- `in_features`: tamaño de cada muestra de entrada
- `out_features`: tamaño de cada muestra de salida
- `bias`: si es True, añade un sesgo aprendible (por defecto True)

Esta es la capa fundamental de las redes totalmente conectadas (fully connected).

In [ ]:
layer1 = nn.Linear(in_features=28*28, out_features=20)
hidden1 = layer1(flat_image)
print(hidden1.size())

### nn.ReLU

Las activaciones no lineales son las que crean los mapeos complejos entre las entradas y salidas del modelo. Se aplican después de transformaciones lineales para introducir no linealidad, ayudando a las redes neuronales a aprender una amplia variedad de fenómenos.

En este modelo, usamos `nn.ReLU` entre nuestras capas lineales, pero hay otras activaciones para introducir no linealidad en tu modelo.

**Función:** $\text{ReLU}(x) = \max(0, x)$

**¿Por qué es importante?**
- Sin activaciones no lineales, múltiples capas lineales se comportarían como una sola capa
- ReLU es computacionalmente eficiente
- Ayuda a mitigar el problema del gradiente desvaneciente

**Alternativas comunes:**
- `nn.Sigmoid`: $\sigma(x) = \frac{1}{1 + e^{-x}}$
- `nn.Tanh`: $\tanh(x) = \frac{e^x - e^{-x}}{e^x + e^{-x}}$
- `nn.LeakyReLU`: Similar a ReLU pero permite pequeños valores negativos

In [ ]:
print(f"Before ReLU: {hidden1}\n\n")
hidden1 = nn.ReLU()(hidden1)
print(f"After ReLU: {hidden1}")

### nn.Sequential

`nn.Sequential` es un contenedor ordenado de módulos. Los datos se pasan a través de todos los módulos en el mismo orden en que se definen. Puedes usar contenedores secuenciales para crear rápidamente una red como `seq_modules`.

**Ventajas:**
- Código más limpio y legible
- Fácil de crear arquitecturas simples
- Los módulos se ejecutan en orden automáticamente

**Cuándo usarlo:**
- Arquitecturas de alimentación directa (feedforward) simples
- Cuando no necesitas lógica condicional en el forward pass

In [ ]:
seq_modules = nn.Sequential(
    flatten,
    layer1,
    nn.ReLU(),
    nn.Linear(20, 10)
)
input_image = torch.rand(3,28,28)
logits = seq_modules(input_image)

### nn.Softmax

La última capa lineal de la red neuronal devuelve **logits** - valores crudos en $[-\infty, \infty]$ - que se pasan al módulo `nn.Softmax`. Los logits se escalan a valores [0, 1] representando las probabilidades predichas del modelo para cada clase. El parámetro `dim` indica la dimensión a lo largo de la cual los valores deben sumar 1.

**Función:** $\text{Softmax}(x_i) = \frac{e^{x_i}}{\sum_j e^{x_j}}$

**Características:**
- Convierte logits en probabilidades
- La suma de todas las probabilidades es 1
- Valores más altos tienen mayor probabilidad

**Nota importante:** Durante el entrenamiento, muchas funciones de pérdida como `CrossEntropyLoss` incluyen Softmax internamente, por lo que no necesitas aplicarlo manualmente.

In [ ]:
softmax = nn.Softmax(dim=1)
pred_probab = softmax(logits)

## Parámetros del Modelo

Muchas capas dentro de una red neuronal están **parametrizadas**, es decir, tienen pesos y sesgos asociados que se optimizan durante el entrenamiento. 

Heredar de `nn.Module` automáticamente rastrea todos los campos definidos dentro de tu objeto modelo, y hace que todos los parámetros sean accesibles usando los métodos `parameters()` o `named_parameters()` de tu modelo.

En este ejemplo, iteramos sobre cada parámetro e imprimimos su tamaño y una vista previa de sus valores.

### ¿Qué son los parámetros?

- **Pesos (weights)**: Los valores que se multiplican por las entradas
- **Sesgos (biases)**: Los valores que se suman después de la multiplicación
- **Entrenables**: Se actualizan durante el entrenamiento mediante backpropagation
- **Inicializados aleatoriamente**: Comienzan con valores pequeños aleatorios

In [ ]:
print(f"Model structure: {model}\n\n")

for name, param in model.named_parameters():
    print(f"Layer: {name} | Size: {param.size()} | Values : {param[:2]} \n")